# Sandro - Prostate158 - PI-CAI: PSA density from the final model

**Inference only** (no training). Runtime > Change runtime type > **T4 GPU**, then **Runtime > Run all**,
and allow Google Drive when asked (twice: once to mount Drive, once to read the bundle).
Keep this tab open while it runs.

* Model: 2d softmax ensemble of seeds 42, 43, 44
  with left-right flip TTA, frozen on the validation split.
* Cohort: PI-CAI v2.0 (Zenodo 6624726), 1440 eligible first studies, axial T2W only.
  This notebook runs folds 2, 3, 4. Each fold (about 5.3 GB) is downloaded from
  Zenodo, checked by MD5, converted, inferred and deleted. Folds 0 and 1 already ran on Sandro's account with this same bundle.
* Expected: about 2-3 h on a T4.
* Results go to `MyDrive/Prostate158_E1_seeds/picai_folds234/` (and a zip next to it). If Colab disconnects, **Run all again**:
  finished folds and finished cases are kept on Drive and skipped.

In [ ]:
import subprocess, torch
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU'
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout)
from google.colab import drive
drive.mount('/content/drive')
import os
OUT_DIR = '/content/drive/MyDrive/Prostate158_E1_seeds/picai_folds234'
FOLDS = [2, 3, 4]
for sub in ('conversions', 'cache_infer', 'predictions'):
    os.makedirs(os.path.join(OUT_DIR, sub), exist_ok=True)


In [ ]:
# Download the bundle (code + 3 frozen checkpoints + locked prespecification + cohort).
import hashlib, io, zipfile
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
BUNDLE_NAME, BUNDLE_SHA256 = 'picai_psad_bundle.zip', '29021394e1c61b3f50420aa8315c3d43eca4e66f8a57398537c0a439361e50bd'
BUNDLE = '/content/' + BUNDLE_NAME
def sha256(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()
if not (os.path.exists(BUNDLE) and sha256(BUNDLE) == BUNDLE_SHA256):
    auth.authenticate_user()
    svc = build('drive', 'v3')
    hits = svc.files().list(q=f"name = '{BUNDLE_NAME}' and trashed = false",
                            fields='files(id, name, size)', includeItemsFromAllDrives=True,
                            supportsAllDrives=True).execute()['files']
    assert hits, 'Bundle not found in your Drive.'
    req = svc.files().get_media(fileId=hits[0]['id'])
    with io.FileIO(BUNDLE, 'wb') as fh:
        dl = MediaIoBaseDownload(fh, req, chunksize=64 << 20)
        done = False
        while not done:
            status, done = dl.next_chunk()
            print(f'  {int(status.progress() * 100)}%', end='')
    print()
assert sha256(BUNDLE) == BUNDLE_SHA256, 'Bundle checksum mismatch'
WORK = '/content/bundle'
zipfile.ZipFile(BUNDLE).extractall(WORK)
os.chdir(WORK)
print('bundle OK:', BUNDLE_SHA256[:16])


In [ ]:
# Labels at the pinned commit (marksheet checked against the local cohort build), and the work folders.
import json, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'SimpleITK==2.4.1', 'nibabel>=5.4'], check=True)
DATA = '/content/picai'
LAB = DATA + '/picai_labels'
os.makedirs(DATA + '/zips', exist_ok=True)
if not os.path.isdir(LAB):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/DIAGNijmegen/picai_labels.git', LAB], check=True)
subprocess.run(['git', '-C', LAB, 'checkout', '-q', 'ce4a4723d7c46d882a6cbaacb40ed6c4be86282f'], check=True)
elig = json.load(open('results/picai_psad/eligibility.json'))
assert sha256(LAB + '/clinical_information/marksheet.csv') == elig['marksheet_sha256'], 'marksheet differs'
for sub in ('conversions', 'cache_infer', 'predictions'):  # finished work lives on Drive
    link = os.path.join(DATA, sub)
    if not os.path.islink(link):
        os.symlink(os.path.join(OUT_DIR, sub), link)
import SimpleITK, nibabel, numpy, scipy
print('labels', 'ce4a472', '| SimpleITK', SimpleITK.Version_VersionString(), '| nibabel',
      nibabel.__version__, '| torch', torch.__version__, '| eligible', elig['n_eligible'])


In [ ]:
# Fold by fold: download + MD5, convert (prepare), infer on the GPU, then free the disk.
import shutil, time
MD5 = {0: 'b0cdd8f5fb7733ca585dca3541586a34', 1: 'd7c8cd0efc0e7f264df8d5acf860662f', 2: '5ac0caf27cfe94dfc04bb408a971b351', 3: '6602bce3d0882d7b9c2153d880c97e96', 4: '8941720d8a8ee66571691cfb10dbee66'}
def md5(p):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 22), b''): h.update(b)
    return h.hexdigest()
def run(args):
    p = subprocess.Popen([sys.executable, '-u', 'scripts/run_picai_psad.py', '--data', DATA] + args,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end='')
    assert p.wait() == 0, 'step failed: Run all again to resume'
for k in FOLDS:
    marker = os.path.join(OUT_DIR, f'fold{k}.done')
    if os.path.exists(marker):
        print(f'fold {k}: done earlier, skipped'); continue
    name = f'picai_public_images_fold{k}.zip'
    z = os.path.join(DATA, 'zips', name)
    for attempt in range(5):
        if os.path.exists(z) and md5(z) == MD5[k]:
            break
        t = time.time()
        subprocess.run(['curl', '-sS', '-L', '--retry', '5', '-C', '-', '-o', z,
                        f'https://zenodo.org/api/records/6624726/files/{name}/content'])
        print(f'fold {k}: downloaded in {time.time() - t:.0f} s')
        if os.path.exists(z) and os.path.getsize(z) > 0 and md5(z) != MD5[k]:
            os.remove(z)
    assert md5(z) == MD5[k], f'fold {k}: MD5 mismatch'
    run(['--phase', 'prepare'])
    run(['--phase', 'infer', '--device', 'cuda'])
    os.remove(z)
    shutil.rmtree(os.path.join(DATA, 'work'), ignore_errors=True)
    open(marker, 'w').write(time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()))
    print(f'fold {k}: done')


In [ ]:
# Copy the summary files next to the per-case results and check the counts.
for f in ('conversion.csv', 'per_case.csv', 'model_identity.json'):
    shutil.copy2(os.path.join('results/picai_psad', f), os.path.join(OUT_DIR, f))
import csv
conv = list(csv.DictReader(open(os.path.join(OUT_DIR, 'conversion.csv'))))
pc = list(csv.DictReader(open(os.path.join(OUT_DIR, 'per_case.csv'))))
ok = sum(r['status'] == 'ok' for r in conv)
print(f"folds {FOLDS}: converted {len(conv)} (ok {ok}); inferred {len(pc)}")
print('geometry check passed:', sum(r['geometry_ok'] == 'True' for r in pc), 'of', len(pc))
zip_path = shutil.make_archive(OUT_DIR, 'zip', OUT_DIR)
print('Done. Results are in', OUT_DIR, 'and', zip_path, f'({os.path.getsize(zip_path) / 1e6:.0f} MB); tell Sandro it finished.')
